# 자료로 따지는 사회 — 시작 노트북

책 [「자료로 따지는 사회」](https://grow.minds.kr/textbooks/css-methods/causal/)의 실습 시작점입니다. 설치는 없습니다. 각 셀을 위에서부터 실행(▶ 또는 Shift+Enter)하세요.

이 노트북은 모의 세계 "여정과 의미"의 데이터를 불러와, 책이 인용하는 첫 수치 두 개를 직접 재현합니다. 나머지 실습은 `data/`의 일곱 벌과 저장소의 스크립트로 이어 갑니다.

## 1. 데이터 불러오기

일곱 벌 중 두 벌(실험판·설문판)을 저장소에서 바로 읽습니다. Colab에는 pandas·numpy·scipy가 이미 있습니다.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/dataminds/css-methods-causal-code/main/data'
exp = pd.read_csv(f'{BASE}/journey_exp.csv')
svy = pd.read_csv(f'{BASE}/journey_svy.csv')
print('실험판:', exp.shape, '| 설문판:', svy.shape)
exp[['id', 'age', 'gender', 'cond', 'mil_t1', 'hjs', 'mil']].head(3)

첫 3행이 책 2장의 기대 출력과 같아야 합니다(1번 참여자 = 36세·개입 집단·기저 3.75·사후 hjs 4.857·mil 2.75).

## 2. 첫 수치 재현 (1): 여정과 의미의 상관 .63

설문판을 정제(주의 점검 통과자만)한 뒤, 5장의 출발 사실인 상관계수를 구합니다.

In [ ]:
svy_clean = svy[svy['attn_1'] == 1]        # 590 -> 566
r = svy_clean['hjs'].corr(svy_clean['mil'])
print(f'정제 후 n = {len(svy_clean)}')
print(f'r(여정, 의미) = {r:.2f}   # 책 5장 = .63')

## 3. 첫 수치 재현 (2): 실험의 주 분석

개입 집단과 통제 집단의 사후 삶의 의미를 비교합니다(시나리오 S1의 심장). 검정을 두 경로로 재는 것이 이 책의 검증 규율입니다 — t검정 하나만 믿지 않고, 조건 딱지를 뒤섞어 만든 "효과 없는 세계"의 분포와도 견줍니다.

In [ ]:
import numpy as np
from scipy import stats

d = exp[exp['attn_1'] == 1]                 # 380 -> 366
tg = d[d['cond'] == 1]['mil']
cg = d[d['cond'] == 0]['mil']
obs = tg.mean() - cg.mean()
t = stats.ttest_ind(tg, cg)
print(f'개입 {tg.mean():.2f} vs 통제 {cg.mean():.2f}  차이 {obs:.3f}')
print(f't = {t.statistic:.2f}, p = {t.pvalue:.3f}   # 책 7장 = t 2.21, p .028')

# 검증 = 독립 재계산: 뒤섞기 검정 (다른 코드 경로)
rng = np.random.default_rng(73)             # 씨앗 73 = 정본
cond = d['cond'].values; y = d['mil'].values
diffs = np.array([(y[(s := rng.permutation(cond)) == 1].mean() - y[s == 0].mean())
                  for _ in range(5000)])
p_perm = (np.abs(diffs) >= abs(obs)).mean()
print(f'뒤섞기 p = {p_perm:.3f}   # 책 7장 = .030 (t검정과 근사 일치)')

## 다음으로

- 일곱 벌 전체와 변수 사전: `data/` 폴더와 `data/README.md`
- 책의 **모든** 수치를 한 번에 재현·검증: `python check_book_claims.py`
- 씨앗을 바꿔 "다른 세계" 만들기(6장 표본 요동): `python make_data_family.py`
- 검증 로그 양식·아홉 단계 체크리스트: `worksheets/`

재현성 규약: 이 책의 수치는 모두 씨앗 73 세계의 것입니다. 화면 숫자가 책과 다르면 먼저 씨앗을 의심하세요.